In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

In [2]:
class Discriminator(nn.Module):
  def __init__(self,img_dim):
    super().__init__()
    self.disc=nn.Sequential(
        nn.Linear(img_dim,128),
        nn.LeakyReLU(0.1),
        nn.Linear(128,1),
        nn.Sigmoid()
    )
  def forward(self,x):
    return self.disc(x)

In [3]:
class Generator(nn.Module):
  def __init__(self,z_dim,img_dim):
    super().__init__()
    self.gen=nn.Sequential(
        nn.Linear(z_dim,256),
        nn.LeakyReLU(0.1),
        nn.Linear(256,img_dim),
        nn.Tanh()
    )
  def forward(self,x):
    return self.gen(x)

In [4]:
device='cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
lr=3e-4
z_dim=64
img_dim=28*28*1
batch_size=32
epochs=58

In [6]:
disc=Discriminator(img_dim).to(device)
gen=Generator(z_dim,img_dim).to(device)
fixed_noi=torch.randn((batch_size,z_dim)).to(device)

In [7]:
transforms=transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5,),(0.5,))
    ]
)

In [8]:
dataset=datasets.MNIST(root='dataset/',transform=transforms,download=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 480kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.52MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.1MB/s]


In [9]:
loader=DataLoader(dataset,batch_size=batch_size,shuffle=True)

In [10]:
opt_disc=optim.Adam(disc.parameters(),lr=lr)
opt_gen=optim.Adam(gen.parameters(),lr=lr)

In [11]:
loss=nn.BCELoss()
writer_fake=SummaryWriter(f'runs/GAN_MNIST/fake')
writer_real=SummaryWriter(f'runs/GAN_MNIST/real')
step=0

In [ ]:
for epoch in range(epochs):
  for batch_idx,(real,_) in enumerate(loader):
    real=real.view(-1,784).to(device)
    batch_size=real.shape[0]

    noise=torch.randn(batch_size,z_dim).to(device)
    fake=gen(noise)
    disc_real=disc(real).view(-1)
    lossD_real=loss(disc_real,torch.ones_like(disc_real))
    disc_fake=disc(fake).view(-1)
    lossD_fake=loss(disc_fake,torch.zeros_like(disc_fake))
    lossD=(lossD_real+lossD_fake)/2
    disc.zero_grad()
    lossD.backward(retain_graph=True)
    opt_disc.step()

    output=disc(fake).view(-1)
    lossG=loss(output,torch.ones_like(output))
    gen.zero_grad()
    lossG.backward()
    opt_gen.step()

    if batch_idx==0:
      print(f'Epoch [{epoch}/{epochs}] \ '
        f'Loss D:{lossD:.4f},Loss G:{lossG:.4f}'
      )
    with torch.no_grad():
      fake=gen(fixed_noi).reshape(-1,1,28,28)
      data=real.reshape(-1,1,28,28)
      img_grid_fake=torchvision.utils.make_grid(fake,normalize=True)
      img_grid_real=torchvision.utils.make_grid(data,normalize=True)

      writer_fake.add_image("Mnist Fake IMage",img_grid_fake,global_step=step)
      writer_real.add_image('Mnist Real IMage',img_grid_real,global_step=step)
      step+=1

<>:24: SyntaxWarning: invalid escape sequence '\ '
<>:24: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-2273619623.py:24: SyntaxWarning: invalid escape sequence '\ '
  print(f'Epoch [{epoch}/{epochs}] \ '


Epoch [0/58] \ Loss D:0.7417,Loss G:0.7033
Epoch [1/58] \ Loss D:0.7824,Loss G:0.7795
Epoch [2/58] \ Loss D:0.3260,Loss G:1.4603
Epoch [3/58] \ Loss D:0.4296,Loss G:1.1213
